# HeartMuLa instrumentals (Colab T4)

Plain [HeartMuLa-oss-3B](https://github.com/HeartMuLa/heartlib) text-to-music, set up the same way as the ACE-Step notebook so you can compare the two:

- **Output:** one generation per track at the model's maximum of 4 minutes, saved exactly as HeartMuLa writes it. There's no processing.
- **Settings:** the model's own defaults: top-k 50, temperature 1.0, CFG 1.5, bfloat16 for HeartMuLa and float32 for HeartCodec.
- **Model:** the version heartlib's README recommends, `HeartMuLa-oss-3B-happy-new-year` with `HeartCodec-oss-20260123`.

**Things to know before you use it:**
- **There's no official instrumental mode.** HeartMuLa is built for songs with lyrics, and empty lyrics crash it. This notebook uses a workaround tested by the community ([heartlib#16](https://github.com/HeartMuLa/heartlib/issues/16)): section tags separated by `<||>` instead of words. The result is mostly instrumental, with the odd "ooh" or "aah", and solo-instrument pieces don't work well.
- **Tags:** HeartMuLa weights **genre** far more than instruments. Tags are comma-separated with no spaces after the commas, e.g. `jazz,piano,relaxing`.
- **Watermark:** the output carries an inaudible watermark ([paper §9](https://arxiv.org/pdf/2601.10547)).
- **Size and memory:** the checkpoints are about 22 GB. The model and codec don't fit on a T4 together, so heartlib's `lazy_load` loads one at a time. That adds a load step to every track.

**Before you start:** Runtime → Change runtime type → **T4 GPU**. Use a fresh session, separate from the ACE-Step notebook, because this one installs different torch versions.

## 1. Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!df -h /content | tail -1

## 2. Hear HeartMuLa's own demos

These come from [heartmula.github.io](https://heartmula.github.io/). They're songs with vocals, the model's main use, and each one is paired with ACE-Step's version of the same song from HeartMuLa's own comparison. You can run this cell before the install.

In [ ]:
from IPython.display import Audio, display
DEMO = "https://heartmula.github.io/static/audios"
for n in (7, 14, 15, 17):
    for model in ("heartmula", "ace-step"):
        print(f"English demo {n}: {model}")
        display(Audio(url=f"{DEMO}/{model}/english/{n}.mp3"))

## 3. Install heartlib

This installs the versions heartlib pins: torch 2.6, torchtune 0.4.0, torchao 0.9.0 and transformers 4.57.0. Colab ships torch 2.11, which these don't support. It skips the pins that would break Colab itself (`numpy==2.0.2` has no Python 3.13 build, and `ipykernel`/`traitlets` are Colab's own kernel).

It downloads about 3 GB of packages. If Colab asks you to **Restart session** at the end, do that, then carry on from step 4.

In [ ]:
!pip install -q torch==2.6.0 torchaudio==2.6.0 torchvision==0.21.0
!pip install -q torchao==0.9.0 torchtune==0.4.0 transformers==4.57.0 tokenizers==0.22.1 einops==0.8.1 vector-quantize-pytorch==1.27.15 accelerate==1.12.0 soundfile
!pip install -q --no-deps git+https://github.com/HeartMuLa/heartlib.git
import importlib.metadata as md
for pkg in ["heartlib", "torch", "torchaudio", "torchtune", "torchao", "transformers"]:
    print(f"{pkg:13s}", md.version(pkg))

## 4. Download the checkpoints (~22 GB)

This uses the files and folder layout from the heartlib README. Add your Hugging Face token as a Colab secret named `HF_TOKEN` (the 🔑 icon) for a faster download. With `CACHE_ON_DRIVE` ticked, the checkpoints are kept on Google Drive so the next session skips the download. That needs 22 GB free on Drive.

In [ ]:
import os
from pathlib import Path
from huggingface_hub import snapshot_download

CACHE_ON_DRIVE = False  #@param {type:"boolean"}
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    print("No HF_TOKEN secret; downloading anonymously.")

if CACHE_ON_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    CKPT = Path("/content/drive/MyDrive/heartmula_ckpt")
else:
    CKPT = Path("/content/heartmula_ckpt")

snapshot_download("HeartMuLa/HeartMuLaGen", local_dir=CKPT)
snapshot_download("HeartMuLa/HeartMuLa-oss-3B-happy-new-year", local_dir=CKPT / "HeartMuLa-oss-3B")
snapshot_download("HeartMuLa/HeartCodec-oss-20260123", local_dir=CKPT / "HeartCodec-oss")
print(sorted(p.name for p in CKPT.iterdir()))

## 5. Set up the pipeline

- **`MULA_DTYPE`:**
  - `bfloat16` is HeartMuLa's default and native precision. A T4 has no bfloat16 hardware, so it runs slower.
  - `float16` is faster on a T4, but it isn't what the model was trained in.
- **Codec:** always runs in float32, as heartlib recommends ("bf16 for HeartCodec may result in the degradation of audio quality").
- **`lazy_load=True`:** loads each model only while it's needed, because both won't fit on a T4 together.

In [ ]:
import torch
from heartlib import HeartMuLaGenPipeline

MULA_DTYPE = "bfloat16"  #@param ["bfloat16", "float16"]
pipe = HeartMuLaGenPipeline.from_pretrained(
    str(CKPT),
    device={"mula": torch.device("cuda"), "codec": torch.device("cuda")},
    dtype={"mula": getattr(torch, MULA_DTYPE), "codec": torch.float32},
    version="3B",
    lazy_load=True,
)
print("Ready. bf16 supported on this GPU:", torch.cuda.is_bf16_supported())

## 6. Prompts

Put one tag line per track in `TAGS`. Start with the genre, then the instruments, mood and tempo, all comma-separated with no spaces after the commas. The examples cover the same styles as the ACE-Step notebook's prompts, so you can compare the results directly.

`LYRICS` is the community instrumental workaround from [heartlib#16](https://github.com/HeartMuLa/heartlib/issues/16). `structure` gives the piece sections; `plain` is the minimal version. Both were reported to avoid lyrics.

In [ ]:
TAGS = """
jazz,piano,bass,drums,relaxing,calm,slow
jazz,piano,keyboard,bass,chill,warm,mellow
jazz,saxophone,piano,bass,romantic,night
latin,acoustic guitar,flute,warm,relaxed
soul,keyboard,bass,drums,groove,relaxed
ambient,piano,strings,peaceful,calm
""".strip().splitlines()
LYRICS = "structure"  #@param ["structure", "plain"]
DURATION = 240  #@param {type:"slider", min:30, max:240, step:10}
SEED = ""  #@param {type:"string"}
SAVE_TO_DRIVE = True  #@param {type:"boolean"}

LYRICS_TEXT = {
    "structure": "[intro]\n<||>\n[verse]\n<||>\n[chorus]\n<||>\n[verse]\n<||>\n[chorus]\n<||>\n[bridge]\n<||>\n[chorus]\n<||>",
    "plain": "<||>\n<||>\n<||>\n<||>\n<||>\n<||>",
}[LYRICS]
print(f"{len(TAGS)} track(s) of up to {DURATION}s, lyrics={LYRICS}")

## 7. Generate

Each track is saved as HeartMuLa's own 48 kHz WAV, with a `.txt` next to it recording the tags and seed. With `SAVE_TO_DRIVE` on, the files go to `MyDrive/heartmula_tracks/`. The player uses an MP3 copy to keep the notebook light; the WAV is the real output.

HeartMuLa writes the music frame by frame and can decide to end the piece before `DURATION`.

In [ ]:
import random, subprocess, time
from datetime import datetime
from IPython.display import Audio, display

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    out_dir = Path("/content/drive/MyDrive/heartmula_tracks")
else:
    out_dir = Path("/content/heartmula_tracks")
out_dir.mkdir(parents=True, exist_ok=True)

for n, tags in enumerate(TAGS, 1):
    seed = int(SEED) + n - 1 if SEED.strip() else random.SystemRandom().randrange(2 ** 31)
    torch.manual_seed(seed)          # heartlib samples with torch's RNG; this only fixes the seed
    name = f"{datetime.now():%Y%m%d-%H%M%S}_{n:02d}_seed{seed}"
    wav = out_dir / f"{name}.wav"
    t0 = time.time()
    with torch.no_grad():            # as in heartlib's examples/run_music_generation.py
        pipe({"lyrics": LYRICS_TEXT, "tags": tags},
             max_audio_length_ms=DURATION * 1000, save_path=str(wav),
             topk=50, temperature=1.0, cfg_scale=1.5)
    (out_dir / f"{name}.txt").write_text(
        f"tags: {tags}\nlyrics: {LYRICS_TEXT!r}\nseed: {seed}\nmula_dtype: {MULA_DTYPE}\n"
        f"max_duration: {DURATION}\ntopk: 50\ntemperature: 1.0\ncfg_scale: 1.5\n")
    print(f"[{n}/{len(TAGS)}] {time.time() - t0:.0f}s | seed {seed}\n  {tags}")
    preview = Path("/content/preview.mp3")
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", str(wav), "-b:a", "256k", str(preview)], check=True)
    display(Audio(str(preview)))
print("Saved in", out_dir)